Web Crawler data flow: 

1. Take seed URL from frontier and request IP from DNS
2. Fetch HTML from external server using IP
3. Extract text data from the HTML.
4. Store the text data in a database.
5. Extract any linked URLs from the web pages and add them to the list (Frontier Queue) of URLs to crawl.
6. Repeat steps 1-5 until all URLs have been crawled.

In [5]:
TESTING_SEED_URL = "https://softwarica.edu.np/courses"
TESTING_ALLOWED_DOMAIN = "softwarica.edu.np"

TSTING_CRAWL_DELAY_SECONDS = 5*24*60*60      
TESTING_MAX_PAGES = 100               
TESTING_USER_AGENT = "SoftwaricaVerticalSearchBot/1.0 (+educational IR project)"

In [ ]:
MAIN_SEED_URL = "https://pureportal.coventry.ac.uk/en/organisations/centre-for-healthcare-and-community-transformation/publications/"
MAIN_ALLOWED_DOMAIN = "pureportal.coventry.ac.uk"
MAIN_CRAWL_DELAY_SECONDS = 24*60*60*30*3    # 3 months      
MAIN_MAX_PAGES = 100               
MAIN_USER_AGENT = "CoventryVerticalSearchBot/1.0 (+educational IR project)"

Ensuring Politness steps: 

1. To make this clear, the steps would be:
2. Fetch the `robots.txt` file for the domain.
3. Parse the `robots.txt` file and store it in the database (MongoDB).
4. When we pull a URL off the queue, check the rules stored in the database (MongoDB) for that domain.
5. If the URL is disallowed, ack the message and move on to the next URL.
6. If the URL is allowed, check the `Crawl-delay` directive.

IF Crawler is failed

7. If the Crawl-delay time has not passed since the last crawl, use ChangeMessageVisibility to extend the visibility timeout and defer reprocessing.
8. If the Crawl-delay time has passed, crawl the page and update the last crawl time for the domain.

In [28]:
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser
import requests

In [29]:
base_url = MAIN_SEED_URL
USER_AGENT = MAIN_USER_AGENT

parsed_url = urlparse(base_url)     # Example: ParseResult(scheme='https', netloc='softwarica.edu.np', path='/courses', params='', query='', fragment='')
robots_url = f"{parsed_url.scheme}://{parsed_url.netloc}/robots.txt"
rfp = RobotFileParser()
rfp.set_url(robots_url)

try: 
    res = requests.get(robots_url, headers={"User-Agent": USER_AGENT}, timeout=10)
    if res.status_code == 200: 
        rfp.parse(res.text.splitlines())
    else: 
        rfp = None
except BaseException as err: 
    print(f"Error: {err}")
    rfp = None
    
print(rfp)

User-agent: *
Crawl-delay: 5
Disallow: /%2A%3F%2Aformat%3Drss
Disallow: /%2A%3F%2Aexport%3Dxls


In [15]:
parsed_url

ParseResult(scheme='https', netloc='softwarica.edu.np', path='/courses', params='', query='', fragment='')

In [33]:
# fetching the content of the robots.txt file 

def fetch_robots(base_url, USER_AGENT):
    parsed_url = urlparse(base_url)     # Example: ParseResult(scheme='https', netloc='softwarica.edu.np', path='/courses', params='', query='', fragment='')
    robots_url = f"{parsed_url.scheme}://{parsed_url.netloc}/robots.txt"
    rfp = RobotFileParser()
    rfp.set_url(robots_url)

    try: 
        res = requests.get(robots_url, headers={"User-Agent": USER_AGENT}, timeout=10)
        if res.status_code == 200: 
            rfp.parse(res.text.splitlines())
        else: 
            rfp = None
    except BaseException as err: 
        print(f"Error: {err}")
        rfp = None
        
    return rfp

In [34]:
robots_data = fetch_robots(TESTING_SEED_URL, TESTING_USER_AGENT)
print(robots_data)

User-agent: Amazonbot
Disallow: /

User-agent: Applebot-Extended
Disallow: /

User-agent: Bytespider
Disallow: /

User-agent: CCBot
Disallow: /

User-agent: ClaudeBot
Disallow: /

User-agent: CloudflareBrowserRenderingCrawler
Disallow: /

User-agent: Google-Extended
Disallow: /

User-agent: GPTBot
Disallow: /

User-agent: meta-externalagent
Disallow: /

User-agent: *
Allow: /


Note: Rate limiting (avoiding system crash by requesting to crawl) is important. Sliding window algorithm can be used to track the number of requests per domain per second